# 18. Q4 Object Ontology Runtime Demo v01

Mục tiêu notebook:

- Load bộ artifact ontology hướng đối tượng đã thiết kế thủ công.
- Thử runtime nhỏ theo flow:

```text
user question
-> intent parser
-> method selector
-> ontology validator
-> SQL template renderer
-> SQL executor
```

Nguyên tắc:

- LLM không sinh SQL tự do trong notebook này.
- SQL chỉ sinh từ `sql_templates.json`.
- Nếu câu hỏi ngoài scope ontology thì reject trước khi SQL.
- Đây là bản demo/inference nhỏ để kiểm tra ontology có vận hành được không.

## 0. Config

Input:

- ontology artifacts
- SQLite database

Output:

- các path runtime dùng trong notebook

In [ ]:
import json
import re
import sqlite3
import time
from pathlib import Path

import pandas as pd


workspace_root = Path("/home/baond12/Documents/Research_Cadshouse_Analytics/benchmarks_baond_exp")
ontology_root = workspace_root / "ontology" / "vietnamese_admin_units_object_ontology_v01"

sqlite_path = Path("/home/baond12/Downloads/file_rndana/vietnamese_admin_units.sqlite")

database_id = "vietnamese_admin_units"

print("ontology_root:", ontology_root)
print("sqlite_path:", sqlite_path)
print("ontology_exists:", ontology_root.exists())
print("sqlite_exists:", sqlite_path.exists())

## 1. Load Ontology Artifacts

Input: JSON files trong `ontology_root`.

Output: dictionary artifacts dùng cho runtime.

In [ ]:
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def load_ontology_artifacts(ontology_root):
    return {
        "classes": load_json(ontology_root / "ontology_classes.json"),
        "properties": load_json(ontology_root / "ontology_properties.json"),
        "relations": load_json(ontology_root / "ontology_relations.json"),
        "methods": load_json(ontology_root / "ontology_methods.json"),
        "intent_rules": load_json(ontology_root / "intent_method_rules.json"),
        "sql_templates": load_json(ontology_root / "sql_templates.json"),
        "unsupported_scope": load_json(ontology_root / "unsupported_scope.json"),
        "runtime_config": load_json(ontology_root / "ontology_runtime_config.json"),
    }


artifacts = load_ontology_artifacts(ontology_root)

method_by_id = {
    item["method_id"]: item
    for item in artifacts["methods"]["methods"]
}

sql_template_by_id = {
    item["sql_template_id"]: item
    for item in artifacts["sql_templates"]["sql_templates"]
}

print("classes:", len(artifacts["classes"]["classes"]))
print("properties:", len(artifacts["properties"]["properties"]))
print("relations:", len(artifacts["relations"]["relations"]))
print("methods:", len(method_by_id))
print("sql_templates:", len(sql_template_by_id))

## 2. Intent Parser Đơn Giản

Input: user question.

Output: `parsed_intent`.

Ghi chú:

- Đây là parser rule-based tối giản để demo ontology.
- Sau này có thể thay bằng LLM function calling, nhưng output vẫn phải map về method ontology.

In [ ]:
def normalize_text(text):
    text = str(text or "").lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text


def contains_any(text, keywords):
    return any(keyword in text for keyword in keywords)


def extract_limit(text):
    match = re.search(r"top\s*(\d+)", text)
    if match:
        return int(match.group(1))

    match = re.search(r"(\d+)\s*(đầu|lớn nhất|nhiều nhất|ít nhất)", text)
    if match:
        return int(match.group(1))

    if contains_any(text, ["nhiều nhất", "ít nhất"]):
        return 1

    return None


def extract_code(text):
    match_5 = re.search(r"\b\d{5}\b", text)
    if match_5:
        return match_5.group(0)

    match_2 = re.search(r"\b\d{2}\b", text)
    if match_2:
        return match_2.group(0)

    return None


def detect_administrative_unit_type(text):
    if "thành phố trực thuộc trung ương" in text:
        return "Thành phố trực thuộc trung ương"
    if "đặc khu" in text:
        return "Đặc khu tại hải đảo"
    if "phường" in text:
        return "Phường"
    if "xã" in text:
        return "Xã"
    if "tỉnh" in text:
        return "Tỉnh"
    return None


def parse_user_intent(question):
    text = normalize_text(question)

    objects = []
    actions = []
    filters = {}

    if contains_any(text, ["vùng hành chính", "vùng địa lý"]):
        objects.append("AdministrativeRegion")

    if contains_any(text, ["loại đơn vị hành chính", "loại hành chính"]):
        objects.append("AdministrativeUnitType")

    if contains_any(text, ["tỉnh", "thành phố trực thuộc trung ương", "cấp tỉnh"]):
        objects.append("Province")

    if contains_any(text, ["phường", "xã", "đặc khu", "cấp xã"]):
        objects.append("CommuneLevelUnit")

    if contains_any(text, ["bao nhiêu", "tổng cộng", "có tất cả"]):
        actions.append("count")

    if contains_any(text, ["liệt kê", "kể tên", "danh sách"]):
        actions.append("list")

    if contains_any(text, ["nhiều nhất", "ít nhất", "top", "xếp hạng"]):
        actions.append("rank")

    if "mã" in text:
        actions.append("find_by_code")

    if "ít nhất" in text:
        filters["order"] = "asc"
    elif contains_any(text, ["nhiều nhất", "top", "xếp hạng"]):
        filters["order"] = "desc"

    administrative_unit_type = detect_administrative_unit_type(text)
    if administrative_unit_type:
        filters["administrativeUnitType"] = administrative_unit_type

    code = extract_code(text)
    if code:
        filters["code"] = code

    limit = extract_limit(text)
    if limit:
        filters["limit"] = limit

    output_property = "fullName"
    if "tên ngắn" in text:
        output_property = "name"

    return {
        "question": question,
        "normalized_question": text,
        "objects": list(dict.fromkeys(objects)),
        "actions": list(dict.fromkeys(actions)),
        "filters": filters,
        "outputProperty": output_property,
    }

## 3. Unsupported Scope Check

Input: user question.

Output:

- unsupported result nếu câu hỏi nằm ngoài dữ liệu/quan hệ hiện tại
- `None` nếu có thể xử lý tiếp

In [ ]:
def detect_unsupported_scope(question, unsupported_scope):
    text = normalize_text(question)

    for scope in unsupported_scope["unsupported_scopes"]:
        for signal in scope.get("signals", []):
            if signal in text:
                return {
                    "scope_id": scope["scope_id"],
                    "reason": scope["reason"],
                    "user_message": scope["user_message"],
                }

    if contains_any(text, ["vùng hành chính", "vùng địa lý", "vùng"]) and contains_any(text, ["tỉnh", "thành phố trực thuộc trung ương"]):
        return {
            "scope_id": "missing_region_province_relation",
            "reason": "No relation links provinces to administrative_regions.",
            "user_message": "Database hiện tại chưa có quan hệ giữa tỉnh/thành phố và vùng hành chính.",
        }

    return None

## 4. Method Selector

Input: `parsed_intent`.

Output: list `method_calls`.

Đây là phần thay thế cho việc LLM tự sinh SQL. Runtime chỉ chọn method ontology.

In [ ]:
def make_method_call(method_id, parameters=None):
    return {
        "method_id": method_id,
        "parameters": parameters or {},
    }


def select_method_calls(parsed_intent):
    objects = parsed_intent["objects"]
    actions = parsed_intent["actions"]
    filters = parsed_intent["filters"]
    output_property = parsed_intent["outputProperty"]

    method_calls = []

    if "AdministrativeRegion" in objects:
        if "count" in actions:
            method_calls.append(make_method_call("AdministrativeRegion.count"))
        if "list" in actions:
            method_calls.append(make_method_call("AdministrativeRegion.list"))

    if "AdministrativeUnitType" in objects:
        if "count" in actions and "Province" not in objects and "CommuneLevelUnit" not in objects:
            method_calls.append(make_method_call("AdministrativeUnitType.count"))
        if "list" in actions and "Province" not in objects and "CommuneLevelUnit" not in objects:
            method_calls.append(make_method_call("AdministrativeUnitType.list"))

    if "Province" in objects and "CommuneLevelUnit" in objects and "rank" in actions:
        params = {
            "order": filters.get("order", "desc"),
            "limit": filters.get("limit", 5),
        }

        child_type = filters.get("administrativeUnitType")
        if child_type in ["Phường", "Xã", "Đặc khu tại hải đảo"]:
            params["childAdministrativeUnitType"] = child_type

        method_calls.append(make_method_call("Province.rankByCommuneLevelUnitCount", params))
        return method_calls

    if "Province" in objects:
        if "find_by_code" in actions and filters.get("code"):
            method_calls.append(make_method_call("Province.findByCode", {"code": filters["code"]}))
        elif "count" in actions:
            params = {}
            if filters.get("administrativeUnitType") in ["Tỉnh", "Thành phố trực thuộc trung ương"]:
                params["administrativeUnitType"] = filters["administrativeUnitType"]
            method_calls.append(make_method_call("Province.count", params))
        elif "list" in actions:
            method_calls.append(make_method_call("Province.list", {"outputProperty": output_property}))

    if "CommuneLevelUnit" in objects and "Province" not in objects:
        if "find_by_code" in actions and filters.get("code"):
            method_calls.append(make_method_call("CommuneLevelUnit.findByCode", {"code": filters["code"]}))
        elif "count" in actions:
            params = {}
            if filters.get("administrativeUnitType") in ["Phường", "Xã", "Đặc khu tại hải đảo"]:
                params["administrativeUnitType"] = filters["administrativeUnitType"]
            method_calls.append(make_method_call("CommuneLevelUnit.count", params))
        elif "list" in actions:
            params = {"outputProperty": output_property}
            if filters.get("administrativeUnitType") in ["Phường", "Xã", "Đặc khu tại hải đảo"]:
                params["administrativeUnitType"] = filters["administrativeUnitType"]
            method_calls.append(make_method_call("CommuneLevelUnit.list", params))

    return method_calls

## 5. Ontology Validator

Input: method calls.

Output:

- valid method calls
- validation errors

In [ ]:
def validate_method_call(method_call, method_by_id):
    method_id = method_call["method_id"]
    parameters = method_call.get("parameters", {})

    if method_id not in method_by_id:
        return {
            "valid": False,
            "reason": f"Unknown method_id: {method_id}",
        }

    method = method_by_id[method_id]
    allowed_param_by_name = {
        item["name"]: item
        for item in method.get("parameters", [])
    }

    for param_name, param_value in parameters.items():
        if param_name not in allowed_param_by_name:
            return {
                "valid": False,
                "reason": f"Parameter {param_name} is not allowed for {method_id}",
            }

        allowed_values = allowed_param_by_name[param_name].get("allowed_values")
        if allowed_values and param_value not in allowed_values:
            return {
                "valid": False,
                "reason": f"Parameter {param_name}={param_value} is not in allowed values {allowed_values}",
            }

    for param in method.get("parameters", []):
        if param.get("required") and param["name"] not in parameters:
            return {
                "valid": False,
                "reason": f"Missing required parameter {param['name']} for {method_id}",
            }

    return {
        "valid": True,
        "reason": "",
    }

## 6. SQL Template Renderer

Input:

- method call
- ontology method catalog
- SQL templates

Output:

- rendered SQL
- bound parameters

In [ ]:
def parameter_key_matches(required_parameter, parameters):
    if "=" in required_parameter:
        name, value = required_parameter.split("=", 1)
        return str(parameters.get(name)) == value
    return required_parameter in parameters


def choose_template_variant(template, parameters):
    variants = template.get("variants", [])
    matched_variants = []

    for variant in variants:
        required_parameters = variant.get("required_parameters", [])
        if required_parameters and all(parameter_key_matches(item, parameters) for item in required_parameters):
            matched_variants.append(variant)

    if matched_variants:
        matched_variants = sorted(
            matched_variants,
            key=lambda item: len(item.get("required_parameters", [])),
            reverse=True,
        )
        return matched_variants[0]

    for variant in variants:
        if not variant.get("required_parameters"):
            return variant

    return None


def render_sql(method_call, method_by_id, sql_template_by_id):
    method = method_by_id[method_call["method_id"]]
    template = sql_template_by_id[method["sql_template_id"]]
    parameters = method_call.get("parameters", {})
    variant = choose_template_variant(template, parameters)

    if not variant:
        return {
            "ok": False,
            "reason": "No SQL template variant matched.",
            "sql": "",
            "sql_params": {},
        }

    sql = variant["sql"]

    if "{order}" in sql:
        order = parameters.get("order", "desc")
        sql = sql.replace("{order}", order.upper())

    if "{limit}" in sql:
        limit = int(parameters.get("limit", 5))
        sql = sql.replace("{limit}", str(limit))

    sql_params = {}

    for key, value in parameters.items():
        if key in ["order", "limit", "outputProperty"]:
            continue
        sql_params[key] = value

    return {
        "ok": True,
        "reason": "",
        "variant_id": variant["variant_id"],
        "sql": sql,
        "sql_params": sql_params,
    }

## 7. SQL Executor

Input: SQL đã render.

Output: DataFrame result.

In [ ]:
def execute_sql(sqlite_path, sql, params=None):
    conn = sqlite3.connect(sqlite_path)
    df = pd.read_sql_query(sql, conn, params=params or {})
    conn.close()
    return df

## 8. Run One Question

Input: một câu hỏi tiếng Việt.

Output:

- parsed intent
- selected method calls
- SQL
- result
- step logs

In [ ]:
def run_ontology_question(question, execute=True):
    logs = []
    started_at = time.time()

    unsupported = detect_unsupported_scope(question, artifacts["unsupported_scope"])
    logs.append({"step": "unsupported_scope_check", "unsupported": unsupported})

    if unsupported:
        return {
            "question": question,
            "status": "unsupported",
            "unsupported": unsupported,
            "logs": logs,
            "elapsed_sec": round(time.time() - started_at, 4),
        }

    parsed_intent = parse_user_intent(question)
    logs.append({"step": "parse_user_intent", "parsed_intent": parsed_intent})

    method_calls = select_method_calls(parsed_intent)
    logs.append({"step": "select_method_calls", "method_calls": method_calls})

    if not method_calls:
        return {
            "question": question,
            "status": "no_method_selected",
            "parsed_intent": parsed_intent,
            "logs": logs,
            "elapsed_sec": round(time.time() - started_at, 4),
        }

    rendered_outputs = []

    for method_call in method_calls:
        validation = validate_method_call(method_call, method_by_id)
        logs.append({"step": "validate_method_call", "method_call": method_call, "validation": validation})

        if not validation["valid"]:
            rendered_outputs.append({
                "method_call": method_call,
                "status": "validation_failed",
                "reason": validation["reason"],
            })
            continue

        rendered = render_sql(method_call, method_by_id, sql_template_by_id)
        logs.append({"step": "render_sql", "method_call": method_call, "rendered": rendered})

        output = {
            "method_call": method_call,
            "status": "rendered" if rendered["ok"] else "render_failed",
            "rendered": rendered,
        }

        if execute and rendered["ok"]:
            result_df = execute_sql(sqlite_path, rendered["sql"], rendered["sql_params"])
            output["result"] = result_df
            output["records"] = result_df.to_dict("records")

        rendered_outputs.append(output)

    return {
        "question": question,
        "status": "ok",
        "parsed_intent": parsed_intent,
        "method_calls": method_calls,
        "outputs": rendered_outputs,
        "logs": logs,
        "elapsed_sec": round(time.time() - started_at, 4),
    }

## 9. Demo Questions

Các câu này lấy từ scope đã bàn:

- count
- list
- composition count + list
- filter by administrative unit type
- rank with join
- unsupported scope

In [ ]:
demo_questions = [
    "Có tất cả bao nhiêu vùng hành chính và kể tên",
    "Có tất cả bao nhiêu loại đơn vị hành chính kể tên",
    "Có tất cả bao nhiêu tỉnh",
    "Có tất cả bao nhiêu phường",
    "Tỉnh nào có nhiều phường nhất và liệt kê top 5",
    "Tên đầy đủ của đơn vị cấp tỉnh có mã 01 là gì?",
    "Tên tiếng Anh của đơn vị hành chính cấp xã có mã 00004 là gì?",
    "Vùng hành chính nào có nhiều tỉnh nhất",
]

for question in demo_questions:
    print("=" * 100)
    print("QUESTION:", question)

    result = run_ontology_question(question)
    print("status:", result["status"])

    if result["status"] == "unsupported":
        print("unsupported:", result["unsupported"])
        continue

    print("parsed_intent:", result.get("parsed_intent"))
    print("method_calls:", result.get("method_calls"))

    for output in result.get("outputs", []):
        print("method_call:", output["method_call"])
        print("status:", output["status"])
        rendered = output.get("rendered", {})
        print("sql:", rendered.get("sql"))
        print("params:", rendered.get("sql_params"))

        if "result" in output:
            display(output["result"].head(10))

## 10. Debug One Question

Dùng cell này khi một câu sai. Xem từng step trước khi vá logic.

In [ ]:
debug_question = "Tỉnh nào có nhiều phường nhất và liệt kê top 5"
debug_result = run_ontology_question(debug_question)

debug_result["logs"]